# 02. Local witness coverage 실습

목표: 합성 trace에서 관찰한 `(이전 skill, routing, 다음 skill)` interface를 수집하고 unseen plan이 요구하는 interface coverage를 계산합니다.

In [ ]:
def interfaces(plan):
    witnesses = set()
    for left, right in zip(plan, plan[1:]):
        left_skill, _ = left
        right_skill, right_route = right
        witnesses.add((left_skill, right_route, right_skill))
    return witnesses

def witness_library(plans):
    result = set()
    for plan in plans:
        result |= interfaces(plan)
    return result

def coverage(plan, library):
    needed = interfaces(plan)
    covered = needed & library
    ratio = len(covered) / len(needed) if needed else 1.0
    return ratio, needed - library

## 고립 원자와 합성 trace 비교

깊이 1의 고립 원자 사례에는 두 모듈이 만나는 interface가 없습니다. 합성 trace는 local witness를 추가합니다.

In [ ]:
atomic_only = [
    [("reverse", "previous")],
    [("rotate", "previous")],
    [("duplicate", "previous")],
]
compound = [
    [("reverse", "previous"), ("rotate", "previous")],
    [("rotate", "previous"), ("duplicate", "previous")],
    [("reverse", "previous"), ("duplicate", "original")],
]
target = [
    ("reverse", "previous"),
    ("rotate", "previous"),
    ("duplicate", "previous"),
]
for name, plans in [("atomic only", atomic_only), ("compound", compound)]:
    ratio, missing = coverage(target, witness_library(plans))
    print(f"{name:12s} coverage={ratio:.0%}, missing={sorted(missing)}")

## RL 탐색을 단순화한 support expansion

현재 library에서 빠진 interface를 포함하고 성공 reward를 받은 rollout만 추가한다고 가정합니다. 실제 RL 최적화가 아니라 논문의 support-enrichment 직관을 보여 주는 진단입니다.

In [ ]:
library = witness_library(compound)
rollouts = [
    ([("duplicate", "previous"), ("reverse", "previous")], 0),
    ([("duplicate", "previous"), ("reverse", "previous")], 1),
    ([("reverse", "previous"), ("rotate", "original")], 1),
]
for plan, reward in rollouts:
    if reward == 1:
        library |= interfaces(plan)

new_target = [("duplicate", "previous"), ("reverse", "previous"), ("rotate", "original")]
ratio, missing = coverage(new_target, library)
print("expanded library:", sorted(library))
print(f"new target coverage={ratio:.0%}, missing={sorted(missing)}")